# Evaluación de impacto de los operativos interinstitucionales (diferencias en diferencias)

**Reproducibilidad del producto HTML.** Este cuaderno recalcula, desde la base de datos original, todas las cifras que aparecen en el HTML (texto, tablas, barras, KPI).

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. Cuando lo pida, sube: `HI_24_26_BUFF1.xlsx` (y, para la verificación final, `261026_impacto_operativos_did_v1.html`).

Resultado: `salida/app_data.json` (los agregados que incrusta el HTML) y `salida/cifras_html.csv` (una fila por cifra).

In [ ]:
# Instalación y carga de datos
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy", "openpyxl"], check=False)

DATA_FILES = ["HI_24_26_BUFF1.xlsx"]
OUT = "salida"; os.makedirs(OUT, exist_ok=True)

try:
    from google.colab import files as _colab_files   # solo existe en Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan and IN_COLAB:
    print("Sube estos archivos:", faltan)
    subidos = _colab_files.upload()          # alternativa: montar Drive y ajustar DATA_PATH
    faltan = [f for f in DATA_FILES if not os.path.exists(f)]
if faltan:
    raise FileNotFoundError(f"Faltan archivos de datos: {faltan}")
DATA_PATH = DATA_FILES[0]
print("Datos listos:", DATA_FILES)

Evaluación de impacto — operativos interinstitucionales en el entorno de centros de diversión nocturna
# Evaluación de impacto de los operativos interinstitucionales (diferencias en diferencias)
**Pregunta.** ¿Cuántos homicidios intencionales menos hubo en el entorno de 300 m de bares, discotecas y
*nightclubs* tras el inicio de los operativos (ago-2025), comparado con lo que habría ocurrido sin ellos?

**Diseño.** Grupo tratado = homicidios dentro del buffer (`BUFF_1 = 1`). Grupo de comparación = homicidios fuera del buffer
(`BUFF_1 = 0`). Se compara el cambio antes/después en ambos grupos (diferencias en diferencias, DiD) con un modelo de Poisson
sobre conteos mensuales. Todo el análisis sale de esta celda en adelante; no hay cifras escritas a mano.

In [ ]:
# Carga y conciliación
import numpy as np, pandas as pd
from scipy import stats

raw = pd.read_excel(DATA_PATH)
d = raw.copy()
d["fecha_infr"] = pd.to_datetime(d["fecha_infr"])
d["m"] = (d.fecha_infr.dt.year - 2024) * 12 + d.fecha_infr.dt.month - 1   # 0 = ene-2024
D0 = 19            # índice de ago-2025 (inicio de los operativos)
M_MAIN = 30        # jul-2026 (último mes completo de la ventana de evaluación)
M_PART = 31        # ago-2026 (mes parcial: la base llega al 29-ago)
MESES = ["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"]
def mlabel(m, short=True):
    y, mm = 2024 + m // 12, m % 12
    return f"{MESES[mm]}-{str(y)[2:]}" if short else f"{MESES[mm]} {y}"

assert set(d.BUFF_1.unique()) <= {0, 1}
filas_ventana = d[(d.m >= 0) & (d.m <= M_MAIN)]
A = {}
A["conciliacion"] = dict(
    filas_brutas=int(len(raw)), fid_unicos=int(raw.FID.nunique()), duplicados_exactos=int(raw.duplicated().sum()),
    fecha_min=str(d.fecha_infr.min().date()), fecha_max=str(d.fecha_infr.max().date()),
    en_buffer=int(d.BUFF_1.sum()), fuera_buffer=int((d.BUFF_1 == 0).sum()),
    pct_en_buffer=float(d.BUFF_1.mean()),
    filas_ventana_modelo=int(len(filas_ventana)),            # ene-24 a jul-26
    filas_ago26_parcial=int((d.m == M_PART).sum()),          # excluidas del modelo principal
    sin_zona=int((d.Admin_Zona.astype(str).str.strip() == "").sum() + d.Admin_Zona.isna().sum()),
)

cnt = d.groupby(["m", "BUFF_1"]).size().unstack(fill_value=0).reindex(range(M_PART + 1), fill_value=0)
cnt.columns = ["fuera", "dentro"]

In [ ]:
# Réplica de la captura (indicador descriptivo: tasa de variación)
def win(a, b): return cnt.loc[a:b].sum()
pre12, post12 = win(7, 18), win(19, 30)          # ago-24..jul-25  vs  ago-25..jul-26
tot_pre, tot_post = int(pre12.sum()), int(post12.sum())
in_pre, in_post = int(pre12["dentro"]), int(post12["dentro"])
out_pre, out_post = int(pre12["fuera"]), int(post12["fuera"])
A["replica"] = dict(
    homicidios=dict(pre=tot_pre, post=tot_post, abs=tot_post - tot_pre, pct=(tot_post - tot_pre) / tot_pre),
    en_buffer=dict(pre=in_pre, post=in_post, abs=in_post - in_pre, pct=(in_post - in_pre) / in_pre, reduccion=1 - in_post / in_pre),
    fuera_buffer=dict(pre=out_pre, post=out_post, abs=out_post - out_pre, pct=(out_post - out_pre) / out_pre),
    participacion=dict(pre=in_pre / tot_pre, post=in_post / tot_post, pp=(in_post / tot_post - in_pre / tot_pre) * 100),
    captura=dict(abs_buffer_captura=-20, abs_buffer_calculada=in_post - in_pre,
                 nota="La captura muestra −20 en la variación absoluta de homicidios en buffer; 45 − 75 = −30."),
)

In [ ]:
# Serie mensual
A["mensual"] = [dict(m=int(m), label=mlabel(m), label_largo=mlabel(m, False), dentro=int(cnt.loc[m, "dentro"]),
                     fuera=int(cnt.loc[m, "fuera"]), post=bool(m >= D0), parcial=bool(m == M_PART))
                for m in range(M_PART + 1)]

In [ ]:
# Motor estadístico: regresión de Poisson (IRLS) con errores robustos agrupados por mes
def poisson_fit(y, X, cluster, maxit=200):
    y = np.asarray(y, float); X = np.asarray(X, float)
    eta = np.full(len(y), np.log(y.mean() + .5)); mu = np.exp(eta); z = eta + (y - mu) / mu; W = mu; b = np.zeros(X.shape[1])
    for _ in range(maxit):
        XtW = X.T * W
        bn = np.linalg.lstsq(XtW @ X, XtW @ z, rcond=None)[0]
        eta = np.clip(X @ bn, -30, 30); mu = np.exp(eta); z = eta + (y - mu) / mu; W = mu
        done = np.max(np.abs(bn - b)) < 1e-10; b = bn
        if done: break
    bread = np.linalg.pinv((X.T * mu) @ X)
    s = (y - mu)[:, None] * X
    cl = np.asarray(cluster); ug = np.unique(cl); G = len(ug)
    S = np.array([s[cl == g].sum(0) for g in ug]); meat = S.T @ S
    N, K = X.shape; V = (G / (G - 1)) * ((N - 1) / (N - K)) * bread @ meat @ bread
    return b, V, G, mu

def coef_out(b, V, G, i):
    se = float(np.sqrt(V[i, i])); t = stats.t.ppf(.975, G - 1)
    return dict(irr=float(np.exp(b[i])), lo=float(np.exp(b[i] - t * se)), hi=float(np.exp(b[i] + t * se)),
                p=float(2 * (1 - stats.t.cdf(abs(b[i] / se), G - 1))), beta=float(b[i]), se=se, clusters=int(G))

def long_panel(ms, c=cnt):
    r = pd.DataFrame([(m, g, int(c.loc[m, "dentro" if g else "fuera"])) for m in ms for g in (0, 1)], columns=["m", "g", "y"])
    r["post"] = (r.m >= D0).astype(int); r["D"] = r.g * r.post
    return r

def did_model(ms, spec="fe", series=None):
    c = cnt if series is None else series
    r = long_panel(ms, c)
    if spec == "2x2":
        X = np.column_stack([np.ones(len(r)), r.g, r.post, r.D]); i = 3
    else:
        M = pd.get_dummies(r.m, dtype=float).values
        X = np.column_stack([r.g.values, M, r.D.values]); i = 1 + M.shape[1]
        if spec == "trend": X = np.column_stack([X, (r.g * r.m).values])
    b, V, G, mu = poisson_fit(r.y, X, r.m)
    return coef_out(b, V, G, i), (b, r, X, i)

In [ ]:
# Modelos principales y de robustez
modelos = []
def add(id_, etiqueta, nota, res, ms, grupo):
    modelos.append(dict(id=id_, etiqueta=etiqueta, nota=nota, grupo=grupo,
                        meses=f"{mlabel(ms[0])} a {mlabel(ms[-1])}", **res))

r1, _ = did_model(range(7, 31), "2x2"); add("A", "Misma ventana que la captura (12 vs 12 meses)", "ago-24 a jul-25 contra ago-25 a jul-26", r1, list(range(7, 31)), 1)
r2, (b2, rr2, X2, i2) = did_model(range(0, 31), "fe"); add("B", "Panel completo: todo el antes disponible", "ene-24 a jul-26, efectos fijos por mes", r2, list(range(0, 31)), 2)
r3, _ = did_model(range(0, 31), "trend"); add("C", "Panel completo + tendencia propia del buffer", "extrapola la tendencia previa del buffer; muy sensible a los datos de 2024", r3, list(range(0, 31)), 2)
r4, _ = did_model(range(0, 32), "fe"); add("D", "Panel completo incluyendo ago-26 (mes parcial)", "la base llega al 29-ago-2026", r4, list(range(0, 32)), 2)
r5, _ = did_model(list(range(0, 5)) + list(range(19, 31)), "fe"); add("E", "Línea base previa al aumento (ene–may 2024)", "compara el posterior con los meses anteriores al salto de jun-24", r5, list(range(0, 5)) + list(range(19, 31)), 3)
A["modelos"] = modelos

# --- Estimaciones centrales y homicidios evitados (modelo B; contrafactual = mismo modelo con efecto = 0)
fe_month = {m: b2[1 + k] for k, m in enumerate(sorted(rr2.m.unique()))}
_, (b1, rr1, X1, i1) = did_model(range(7, 31), "fe")
fe_month1 = {m: b1[1 + k] for k, m in enumerate(sorted(rr1.m.unique()))}
cf = []
for m in range(D0, M_MAIN + 1):
    cf.append(dict(m=m, label=mlabel(m), obs=int(cnt.loc[m, "dentro"]),
                   cf_A=float(np.exp(b1[0] + fe_month1[m])), cf_B=float(np.exp(b2[0] + fe_month[m]))))
A["contrafactual"] = cf
obs_post = int(sum(x["obs"] for x in cf)); cf_post = float(sum(x["cf_B"] for x in cf)); cf_post_A = float(sum(x["cf_A"] for x in cf))
def evit(irr): return obs_post / irr - obs_post       # homicidios evitados si el efecto es 1/irr
A["efecto"] = dict(
    obs_post=obs_post, cf_modelo_A=cf_post_A, cf_modelo_B=cf_post, evitados_B=cf_post - obs_post,
    evitados_A=evit(r1["irr"]), evitados_A_lo=evit(r1["hi"]), evitados_A_hi=evit(r1["lo"]),
    evitados_B_lo=evit(r2["hi"]), evitados_B_hi=evit(r2["lo"]),
    reduccion_A=1 - r1["irr"], reduccion_A_lo=1 - r1["hi"], reduccion_A_hi=1 - r1["lo"],
    reduccion_B=1 - r2["irr"], reduccion_B_lo=1 - r2["hi"], reduccion_B_hi=1 - r2["lo"],
    did_niveles=(in_post - in_pre) - (out_post - out_pre),
    razon_de_razones=(in_post / in_pre) / (out_post / out_pre),
)

In [ ]:
# Estudio de eventos (trimestres relativos a ago-2025) y prueba de tendencias previas
def tbin(m): return (m - D0) // 3
ms_ev = list(range(1, 31))           # feb-24 a jul-26 (ene-24 queda fuera para completar trimestres)
rev = long_panel(ms_ev); rev["b"] = rev.m.map(tbin)
bins = sorted(rev.b.unique()); REF = -1
Bd = pd.get_dummies(rev.b, dtype=float)[bins].values
inter = np.column_stack([rev.g.values * (rev.b.values == k) for k in bins if k != REF])
Xev = np.column_stack([rev.g.values, Bd, inter])
bev, Vev, Gev, _ = poisson_fit(rev.y, Xev, rev.m)
ev = []; first = 1 + len(bins); k_idx = 0
for k in bins:
    mm = [m for m in ms_ev if tbin(m) == k]
    lab = f"{mlabel(mm[0])} – {mlabel(mm[-1])}"
    if k == REF:
        ev.append(dict(bin=int(k), label=lab, irr=1.0, lo=1.0, hi=1.0, ref=True, post=False))
    else:
        o = coef_out(bev, Vev, Gev, first + k_idx); k_idx += 1
        ev.append(dict(bin=int(k), label=lab, irr=o["irr"], lo=o["lo"], hi=o["hi"], p=o["p"], ref=False, post=bool(k >= 0)))
A["evento"] = ev
_post = [e["irr"] for e in ev if e["post"]]; _estable = [e["irr"] for e in ev if (not e["post"]) and (not e["ref"]) and e["bin"] >= -4]
A["evento_resumen"] = dict(post_min=min(_post), post_max=max(_post), estable_min=min(_estable), estable_max=max(_estable))
pre_idx = [first + j for j, k in enumerate([k for k in bins if k != REF]) if k < 0]
bp = bev[pre_idx]; Vp = Vev[np.ix_(pre_idx, pre_idx)]
wald = float(bp @ np.linalg.solve(Vp, bp)); A["tendencias_previas"] = dict(
    wald=wald, gl=len(pre_idx), p=float(1 - stats.chi2.cdf(wald, len(pre_idx))))

pre_idx2 = [first + j for j, k in enumerate([k for k in bins if k != REF]) if -4 <= k < 0]
bp2 = bev[pre_idx2]; Vp2 = Vev[np.ix_(pre_idx2, pre_idx2)]
wald2 = float(bp2 @ np.linalg.solve(Vp2, bp2))
A["tendencias_previas"].update(wald_estable=wald2, gl_estable=len(pre_idx2), p_estable=float(1 - stats.chi2.cdf(wald2, len(pre_idx2))))

# tendencia lineal diferencial en el periodo previo (ene-24 a jul-25)
rp = long_panel(range(0, D0)); Xp = np.column_stack([np.ones(len(rp)), rp.g, rp.m, rp.g * rp.m])
bpp, Vpp, Gpp, _ = poisson_fit(rp.y, Xp, rp.m)
o = coef_out(bpp, Vpp, Gpp, 3)
A["tendencias_previas"].update(pend_dif_mensual=o["beta"], pend_dif_irr_mes=o["irr"], pend_p=o["p"])
# cociente dentro/total por mes antes del salto vs después
A["salto_2024"] = dict(
    dentro_ene_may24=int(cnt.loc[0:4, "dentro"].sum()), total_ene_may24=int(cnt.loc[0:4].sum().sum()),
    dentro_jun24_jul25=int(cnt.loc[5:18, "dentro"].sum()), total_jun24_jul25=int(cnt.loc[5:18].sum().sum()),
    dentro_post=int(cnt.loc[19:30, "dentro"].sum()), total_post=int(cnt.loc[19:30].sum().sum()),
)
for k in ("ene_may24", "jun24_jul25", "post"):
    pass
s = A["salto_2024"]
s["pct_ene_may24"] = s["dentro_ene_may24"] / s["total_ene_may24"]
s["pct_jun24_jul25"] = s["dentro_jun24_jul25"] / s["total_jun24_jul25"]
s["pct_post"] = s["dentro_post"] / s["total_post"]
s["ultimo_idx_previo"] = 4; s["prom_mes_ene_may24"] = s["dentro_ene_may24"] / 5; s["prom_mes_jun24_jul25"] = s["dentro_jun24_jul25"] / 14; s["prom_mes_post"] = s["dentro_post"] / 12

In [ ]:
# Placebos en el tiempo (solo ago-24 a jul-25: periodo previo estable, sin datos de operativos)
plac = []
for k in range(10, 16):      # inicio ficticio entre nov-24 y abr-25
    ms = list(range(7, D0)); r = long_panel(ms); r["D"] = r.g * (r.m >= k).astype(int)
    M = pd.get_dummies(r.m, dtype=float).values; X = np.column_stack([r.g.values, M, r.D.values])
    b, V, G, _ = poisson_fit(r.y, X, r.m); o = coef_out(b, V, G, X.shape[1] - 1)
    plac.append(dict(inicio=mlabel(k), irr=o["irr"], lo=o["lo"], hi=o["hi"], p=o["p"]))
A["placebos"] = plac
A["placebos_resumen"] = dict(min=min(p["irr"] for p in plac), max=max(p["irr"] for p in plac), n=len(plac),
                             n_signif=int(sum(p["p"] < .05 for p in plac)))

In [ ]:
# Sensibilidad: se excluye un mes a la vez (modelo B)
loo = []
for x in range(0, 31):
    ms = [m for m in range(0, 31) if m != x]; o, _ = did_model(ms, "fe"); loo.append((x, o["irr"]))
irrs = [v for _, v in loo]
A["dejar_uno_fuera"] = dict(min=min(irrs), max=max(irrs), mes_min=mlabel(loo[int(np.argmin(irrs))][0], False),
                            mes_max=mlabel(loo[int(np.argmax(irrs))][0], False))

In [ ]:
# Mecanismo: horario nocturno vs diurno y tipo de arma
d["noche"] = d.HORARIO.isin(["NOCHE", "MADRUGADA"]); d["fuego"] = d.arma.eq("ARMA DE FUEGO")
def sub_did(mask, etiqueta):
    c = d[mask].groupby(["m", "BUFF_1"]).size().unstack(fill_value=0).reindex(range(M_PART + 1), fill_value=0)
    for col in (0, 1):
        if col not in c.columns: c[col] = 0
    c = c[[0, 1]]; c.columns = ["fuera", "dentro"]
    oA, _ = did_model(range(7, 31), "2x2", c)
    p0, p1 = c.loc[7:18], c.loc[19:30]
    return dict(etiqueta=etiqueta, dentro_pre=int(p0.dentro.sum()), dentro_post=int(p1.dentro.sum()),
                fuera_pre=int(p0.fuera.sum()), fuera_post=int(p1.fuera.sum()), **oA)
A["mecanismo"] = [sub_did(d.noche, "Noche y madrugada (18:00 a 05:59)"), sub_did(~d.noche, "Mañana y tarde (06:00 a 17:59)"),
                  sub_did(d.fuego, "Arma de fuego"), sub_did(~d.fuego, "Otras armas")]

In [ ]:
# Control más exigente: dentro vs fuera en la misma administración zonal (efectos fijos zona × mes)
dz = d[(d.m >= 0) & (d.m <= M_MAIN)].copy(); dz["z"] = dz.Admin_Zona.astype(str).str.strip()
dz = dz[dz.z != ""]
def zona_mes(m0):
    sub = dz[dz.m >= m0]
    ins_z = sub[sub.BUFF_1 == 1].groupby("z").size(); zonas_ok = ins_z[ins_z >= 5].index.tolist()
    grp = sub.groupby(["z", "m", "BUFF_1"]).size()
    rows = [(z, m, g, int(grp.get((z, m, g), 0))) for z in zonas_ok for m in range(m0, 31) for g in (0, 1)]
    rz = pd.DataFrame(rows, columns=["z", "m", "g", "y"]); rz["D"] = rz.g * (rz.m >= D0)
    tot = rz.groupby(["z", "m"]).y.transform("sum"); rz = rz[tot > 0].reset_index(drop=True)
    ZM = pd.get_dummies(rz.z + "|" + rz.m.astype(str), dtype=float).values
    GZ = pd.get_dummies(rz.z, dtype=float).values * rz.g.values[:, None]
    X = np.column_stack([ZM, GZ, rz.D.values.astype(float)])
    b, V, G, _ = poisson_fit(rz.y, X, rz.z)
    return coef_out(b, V, G, X.shape[1] - 1), zonas_ok, int(ins_z[zonas_ok].sum())
oz1, zonas_ok, ev_z = zona_mes(7); oz0, _, _ = zona_mes(0)
A["zona_mes"] = dict(zonas=zonas_ok, n_zonas=len(zonas_ok), eventos_dentro_ventana12=ev_z, ventana12=oz1, completo=oz0)
modelos.append(dict(id="A2", etiqueta="Dentro vs. fuera en la misma zona (zona × mes), 12 vs 12 meses",
                    nota=f"{len(zonas_ok)} administraciones zonales con ≥5 homicidios en buffer; errores agrupados por zona",
                    grupo=1, meses="ago-24 a jul-26", **oz1))
modelos.append(dict(id="B2", etiqueta="Dentro vs. fuera en la misma zona (zona × mes), panel completo",
                    nota=f"{len(zonas_ok)} administraciones zonales; errores agrupados por zona",
                    grupo=2, meses="ene-24 a jul-26", **oz0))
modelos.sort(key=lambda m: (m["grupo"], m["id"]))
A["modelos"] = modelos

In [ ]:
# Descriptivo territorial: dónde están los homicidios en buffer
zt = d[(d.m >= 7) & (d.m <= 30) & (d.BUFF_1 == 1)].copy(); zt["z"] = zt.Admin_Zona.astype(str).str.strip().replace("", "s/d")
zt["periodo"] = np.where(zt.m >= D0, "post", "pre")
ct = zt.groupby(["z", "periodo"]).size().unstack(fill_value=0)
for c_ in ("pre", "post"):
    if c_ not in ct: ct[c_] = 0
ct = ct.assign(tot=ct.pre + ct.post).sort_values("tot", ascending=False)
A["zonas_buffer"] = [dict(zona=z.title(), pre=int(r.pre), post=int(r.post), total=int(r.tot), pct=float(r.tot / ct.tot.sum()))
                     for z, r in ct.iterrows()]

In [ ]:
# Sensibilidad territorial: se excluye una administración zonal a la vez (ventana del escenario A)
dzl = []
for z in A["zonas_buffer"]:
    nombre = z["zona"]
    mask = d.Admin_Zona.astype(str).str.strip().str.title() != nombre
    c = d[mask].groupby(["m", "BUFF_1"]).size().unstack(fill_value=0).reindex(range(M_PART + 1), fill_value=0)
    c = c[[0, 1]]; c.columns = ["fuera", "dentro"]
    o, _ = did_model(range(7, 31), "2x2", c)
    dzl.append(dict(zona=nombre, **{k: o[k] for k in ("irr", "lo", "hi", "p")}, cambio=o["irr"] - 1, cambio_lo=o["lo"] - 1, cambio_hi=o["hi"] - 1))
A["sin_zona_i"] = dzl
_z = A["zonas_buffer"]
A["zonas_resumen"] = dict(zona_top=_z[0]["zona"], cambio_top=_z[0]["post"] - _z[0]["pre"], cambio_total=sum(z["post"] - z["pre"] for z in _z),
                          n_suben=int(sum(z["post"] > z["pre"] for z in _z)), n_zonas=len(_z),
                          otras_cambio_min=min(x["cambio"] for x in dzl[1:]), otras_cambio_max=max(x["cambio"] for x in dzl[1:]))

In [ ]:
# Resumen y salida
for m_ in A["modelos"]:
    m_["cambio"] = m_["irr"] - 1; m_["cambio_lo"] = m_["lo"] - 1; m_["cambio_hi"] = m_["hi"] - 1
A["mod"] = {m_["id"]: m_ for m_ in A["modelos"]}
for m_ in A["mecanismo"]:
    m_["cambio"] = m_["irr"] - 1; m_["cambio_lo"] = m_["lo"] - 1; m_["cambio_hi"] = m_["hi"] - 1
A["meta"] = dict(titulo="Evaluación de impacto de los operativos interinstitucionales en el entorno de centros de diversión nocturna",
                 inicio_operativos="agosto 2025", idx_inicio=D0, ventana_post="ago-2025 a jul-2026", ventana_pre_comparable="ago-2024 a jul-2025",
                 radio_buffer_m=300, centros_tolerancia=373)
print(pd.DataFrame([{k: m[k] for k in ("id", "etiqueta", "irr", "lo", "hi", "p")} for m in A["modelos"]]).round(3).to_string())
print(pd.DataFrame(A["sin_zona_i"]).round(3).to_string()); print(A["zonas_resumen"]); print(A["efecto"]); print(A["tendencias_previas"]); print(A["zona_mes"]); print(A["salto_2024"]); print(A["dejar_uno_fuera"])
print(pd.DataFrame(A["placebos"]).round(3)); print(pd.DataFrame(A["mecanismo"]).round(3).to_string())
print(pd.DataFrame(A["evento"]).round(3)); print(A["replica"]); print(A["conciliacion"])

In [ ]:
# Tabla maestra: cada cifra de APP_DATA (de aquí salen texto, tablas, barras y KPI del HTML)
import json, pandas as pd

def _aplanar(o, ruta=""):
    if isinstance(o, dict):
        for k, v in o.items(): yield from _aplanar(v, f"{ruta}.{k}" if ruta else k)
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from _aplanar(v, f"{ruta}.{i}")
    else:
        yield ruta, o

cifras = pd.DataFrame(list(_aplanar(A)), columns=["ruta", "valor"])
cifras.to_csv(f"{OUT}/cifras_html.csv", index=False)
json.dump(A, open(f"{OUT}/app_data.json", "w"), ensure_ascii=False, indent=1, default=str)
print(len(cifras), "cifras →", f"{OUT}/cifras_html.csv y app_data.json")
cifras.head(40)

In [ ]:
# Verificación contra el HTML publicado (opcional): sube el .html a Colab para correrla
import re, json, os, math
HTML = "261026_impacto_operativos_did_v1.html"
if not os.path.exists(HTML) and IN_COLAB:
    print("Sube", HTML, "para comparar (o ignora esta celda)"); _colab_files.upload()
if os.path.exists(HTML):
    src = open(HTML, encoding="utf-8").read()
    m = re.search(r'<script id="app-data" type="application/json">(.*?)</script>', src, re.S)
    if not m: raise SystemExit("El HTML no incrusta APP_DATA como <script id='app-data'>")
    H = json.loads(m.group(1))
    def _cmp(a, b, ruta, dif):
        if isinstance(a, dict) and isinstance(b, dict):
            for k in a:
                if k in b: _cmp(a[k], b[k], f"{ruta}.{k}" if ruta else k, dif)
        elif isinstance(a, list) and isinstance(b, list):
            for i, (x, y) in enumerate(zip(a, b)): _cmp(x, y, f"{ruta}.{i}", dif)
        elif isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
            if not math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-9): dif.append((ruta, a, b))
        elif a != b: dif.append((ruta, a, b))
    dif = []; _cmp(A, H, "", dif)
    print("Coinciden todas las cifras comunes" if not dif else f"{len(dif)} diferencias:")
    for d in dif[:50]: print("  ", d)
else:
    print("HTML no encontrado; verificación omitida")